# Session 5: Build and check a small food app

**BGS College workshop · V Naga Sai Rahul · Student notebook**

Your starter contains one announced quantity bug. Keep evidence of the failure, review a small repair, and check the result.

Work in pairs for 75 minutes. Save a copy in your Drive, choose a CPU runtime and run each section in order. No API key is needed for the project checks. You can ask a prepared browser or laptop coding assistant for a patch; record which route you actually used.

You will reproduce a failing test, repair quantity handling, preview the app and add a Clear cart action. All menu items and orders are fictional.


## 1. Open the real starter

This cell checks for Node.js 18.20 or later and writes the supplied project into the runtime. It preserves existing files if you rerun it. A missing or old Node runtime produces a clear error; this notebook does not silently install a coding client, router or model. The downloadable laptop project recommends Node.js 20 or later.

Colab runs on a cloud machine. Its files and `localhost` belong to that machine. Save your notebook and download your project before the temporary runtime ends.


**Prepare the project**

**What:** Check Node and write the supplied starter files without replacing existing edits.

**Why:** The lab needs a known project and unchanged reference tests.

**Expected:** A project folder and runtime version; no automatic installation or model request.

**Read the code:**
1. Check Node before writing any files.
2. Create missing starter files and protect the supplied tests with a fingerprint.
3. Define run_tests so baseline and repaired code use the same checks.


In [ ]:
# @title Prepare the project
# Compare the starter and edited source with a readable before-and-after diff.
import difflib
# Use a file fingerprint to detect changes to the reference tests.
import hashlib
# Escape HTML before putting the app into the preview iframe.
import html
# Read and write the structured evidence as JSON.
import json
# Match the supplied JavaScript module lines and Node test counts.
import re
# Find installed programs such as Node without assuming a fixed location.
import shutil
# Run Node commands and collect their real output.
import subprocess
# Package the completed project into a ZIP file.
import zipfile
# Build file paths that work in the notebook runtime.
from pathlib import Path
# Show the app preview and clickable download links below a cell.
from IPython.display import HTML, FileLink, display

# Find Node in this runtime; this is separate from any optional laptop coding client.
NODE = shutil.which("node")
# Stop early when the runtime has no Node executable.
if NODE is None:
    # Explain the missing prerequisite instead of installing an unknown replacement.
    raise RuntimeError("Node.js 18.20 or later is required. This notebook installs nothing automatically. Ask the instructor for the preflighted runtime or use the downloaded laptop starter.")
# Ask Node for its version and keep the returned text for the evidence record.
node_version = subprocess.run([NODE, "--version"], check=True, capture_output=True, text=True).stdout.strip()
# Turn the version into numbers so an older runtime is rejected correctly.
if tuple(int(part) for part in node_version.lstrip("v").split(".")[:3]) < (18, 20, 0):
    # Ask for the prepared runtime when the installed Node version is too old.
    raise RuntimeError(f"This runtime has {node_version}; the Colab companion requires Node.js 18.20+. Ask the instructor before installing anything.")

# Keep the supplied assets unchanged so the announced bug remains available to investigate.
ASSETS = {'.gitignore': '.env\n.env.*\nnode_modules/\n*.log\n', 'AGENTS.md': '# Workshop project\n\nThis is a synthetic BGS College food-cart exercise. It does not submit an order or payment.\n\n- Work only in this workshop folder.\n- The announced starter bug is quantity handling in cart.js.\n- Keep the existing input validation and tests.\n- Add no dependencies for the initial repair.\n- Tests: `npm test` (Node built-in test runner).\n- App: `npm start`, then open http://127.0.0.1:8085.\n- First propose a brief plan. After editing, show the diff and run the same tests.\n- Report observed output; do not claim an unexecuted test passed.\n- Keep credentials out of files, prompts and screenshots. Ask before unrelated changes.\n', 'app.js': 'import {MENU, calculateCart, orderSummary} from \'./cart.js\';\n\nconst menu = document.querySelector(\'#menu\');\nconst cartLines = document.querySelector(\'#cart-lines\');\nconst total = document.querySelector(\'#total\');\nconst summary = document.querySelector(\'#summary\');\nconst error = document.querySelector(\'#error\');\nconst summaryButton = document.querySelector(\'#summary-button\');\nconst currency = value => new Intl.NumberFormat(\'en-IN\', {style: \'currency\', currency: \'INR\', maximumFractionDigits: 0}).format(value);\n\nfor (const [itemId, item] of Object.entries(MENU)) {\n  const row = document.createElement(\'div\');\n  row.className = \'menu-row\';\n  row.innerHTML = `<div><p class="item-name">${item.name}</p><p class="price">${currency(item.priceInr)} each</p></div><label>Qty <input aria-label="${item.name} quantity" data-item="${itemId}" type="number" min="0" max="10" step="1" value="0"></label>`;\n  menu.append(row);\n}\n\nfunction renderCart() {\n  error.textContent = \'\';\n  summary.textContent = \'\';\n  try {\n    const cart = Object.fromEntries([...menu.querySelectorAll(\'input\')].map(input => [input.dataset.item, Number(input.value)]));\n    const result = calculateCart(cart);\n    cartLines.replaceChildren();\n    if (!result.items.length) {\n      const empty = document.createElement(\'p\');\n      empty.className = \'empty\';\n      empty.textContent = \'Your cart is empty. Choose an item to begin.\';\n      cartLines.append(empty);\n    }\n    for (const item of result.items) {\n      const line = document.createElement(\'div\');\n      line.className = \'cart-line\';\n      line.innerHTML = `<span>${item.quantity} × ${item.name}</span><span>${currency(item.subtotalInr)}</span>`;\n      cartLines.append(line);\n    }\n    total.textContent = currency(result.totalInr);\n    summaryButton.disabled = result.items.length === 0;\n    summaryButton.onclick = () => { summary.textContent = orderSummary(result); };\n  } catch (problem) {\n    error.textContent = problem.message;\n    summaryButton.disabled = true;\n  }\n}\nmenu.addEventListener(\'input\', renderCart);\nrenderCart();\n', 'cart.js': "export const MENU = Object.freeze({\n  dosa: Object.freeze({name: 'Dosa', priceInr: 40}),\n  tea: Object.freeze({name: 'Tea', priceInr: 20}),\n  idli: Object.freeze({name: 'Idli plate', priceInr: 30}),\n});\n\nexport function calculateCart(cart) {\n  if (!cart || Array.isArray(cart) || typeof cart !== 'object') {\n    throw new TypeError('Cart must be an object of item IDs and quantities.');\n  }\n  const items = [];\n  for (const [itemId, quantity] of Object.entries(cart)) {\n    if (!Object.hasOwn(MENU, itemId)) throw new RangeError('Unknown menu item.');\n    if (!Number.isInteger(quantity) || quantity < 0 || quantity > 10) {\n      throw new RangeError('Quantity must be an integer from 0 to 10.');\n    }\n    if (quantity === 0) continue;\n    const item = MENU[itemId];\n    const subtotal = item.priceInr;\n    items.push({itemId, name: item.name, quantity, unitPriceInr: item.priceInr, subtotalInr: subtotal});\n  }\n  return {items, totalInr: items.reduce((total, item) => total + item.subtotalInr, 0), currency: 'INR', simulation: true};\n}\n\nexport function orderSummary(result) {\n  return result.items.length\n    ? result.items.map(item => `${item.quantity} × ${item.name}`).join(', ') + ` — INR ${result.totalInr} (simulation)`\n    : 'Your cart is empty.';\n}\n", 'cart.test.js': "import test from 'node:test';\nimport assert from 'node:assert/strict';\nimport {calculateCart, orderSummary} from './cart.js';\n\ntest('an empty cart totals zero and has a useful summary', () => {\n  const result = calculateCart({});\n  assert.equal(result.totalInr, 0);\n  assert.deepEqual(result.items, []);\n  assert.equal(orderSummary(result), 'Your cart is empty.');\n});\n\ntest('one dosa and one tea use their stated prices', () => {\n  const result = calculateCart({dosa: 1, tea: 1});\n  assert.equal(result.totalInr, 60);\n  assert.equal(result.currency, 'INR');\n  assert.equal(result.simulation, true);\n});\n\ntest('quantity contributes to each subtotal and total', () => {\n  const result = calculateCart({dosa: 2, tea: 1});\n  assert.equal(result.items[0].subtotalInr, 80);\n  assert.equal(result.totalInr, 100);\n  assert.equal(orderSummary(result), '2 × Dosa, 1 × Tea — INR 100 (simulation)');\n});\n\ntest('zero quantity removes a line', () => {\n  assert.deepEqual(calculateCart({dosa: 0}).items, []);\n});\n\ntest('unknown item IDs are rejected', () => {\n  assert.throws(() => calculateCart({burger: 1}), /Unknown menu item/);\n});\n\ntest('negative, fractional, excessive and nonnumeric quantities are rejected', () => {\n  for (const quantity of [-1, 1.5, 11, '2', true, null, NaN]) {\n    assert.throws(() => calculateCart({dosa: quantity}), /Quantity must/);\n  }\n});\n\ntest('invalid cart containers are rejected', () => {\n  for (const cart of [null, [], 'dosa', 1]) {\n    assert.throws(() => calculateCart(cart), /Cart must/);\n  }\n});\n", 'CLAUDE.md': '# Workshop project\n\nThis is a synthetic BGS College food-cart exercise. It does not submit an order or payment.\n\n- Work only in this workshop folder.\n- The announced starter bug is quantity handling in cart.js.\n- Keep the existing input validation and tests.\n- Add no dependencies for the initial repair.\n- Tests: `npm test` (Node built-in test runner).\n- App: `npm start`, then open http://127.0.0.1:8085.\n- First propose a brief plan. After editing, show the diff and run the same tests.\n- Report observed output; do not claim an unexecuted test passed.\n- Keep credentials out of files, prompts and screenshots. Ask before unrelated changes.\n', 'GEMINI.md': '# Workshop project\n\nThis is a synthetic BGS College food-cart exercise. It does not submit an order or payment.\n\n- Work only in this workshop folder.\n- The announced starter bug is quantity handling in cart.js.\n- Keep the existing input validation and tests.\n- Add no dependencies for the initial repair.\n- Tests: `npm test` (Node built-in test runner).\n- App: `npm start`, then open http://127.0.0.1:8085.\n- First propose a brief plan. After editing, show the diff and run the same tests.\n- Report observed output; do not claim an unexecuted test passed.\n- Keep credentials out of files, prompts and screenshots. Ask before unrelated changes.\n', 'index.html': '<!doctype html>\n<html lang="en">\n<head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>BGS Food Lab</title><link rel="stylesheet" href="styles.css"></head>\n<body>\n  <header><a class="brand" href="/">BGS Food Lab</a><span>Session 5 · Coding with AI</span></header>\n  <main>\n    <section class="intro"><p class="eyebrow">YOUR WORKSHOP BUILD</p><h1>A little menu.<br>A useful code change.</h1><p>Choose a few items, inspect the total and keep the evidence behind your repair.</p></section>\n    <div class="workspace">\n      <section aria-labelledby="menu-title"><div class="section-title"><h2>Today’s menu</h2><span>Prices in INR</span></div><div id="menu"></div><p class="hint">Quantities from 0 to 10. All items and orders are fictional.</p></section>\n      <aside aria-labelledby="cart-title"><p class="eyebrow">YOUR CART</p><h2 id="cart-title">Order summary</h2><div id="cart-lines" aria-live="polite"></div><div class="total"><span>Total</span><strong id="total">₹0</strong></div><p id="error" role="alert"></p><button id="summary-button" type="button">Show simulated order</button><p id="summary" aria-live="polite"></p><p class="hint">This exercise creates a draft on this page. It sends no order or payment.</p></aside>\n    </div>\n  </main>\n  <footer>V Naga Sai Rahul · BGS College workshop · Learn the behavior, review the diff, run the tests.</footer>\n  <script type="module" src="app.js"></script>\n</body></html>\n', 'package.json': '{\n  "name": "bgs-food-lab",\n  "private": true,\n  "type": "module",\n  "scripts": {"start": "node server.mjs", "test": "node --test"},\n  "engines": {"node": ">=20"}\n}\n', 'README.md': '# BGS Food Lab\n\nA small local food-cart app for Session 5. The menu and orders are fictional. Nothing is submitted or paid.\n\n## Before the session\n\nUse Node.js 20 or later, a modern browser and one instructor-preflighted coding-assistant route. This project has no npm dependencies; `npm install` is not required.\n\nRun `node --version`, then open a terminal in this starter folder.\n\n## Run the app and baseline\n\n```text\nnpm test\nnpm start\n```\n\nOpen http://127.0.0.1:8085. Keep the app terminal open; use a second terminal for tests or the coding assistant. The development server binds only to 127.0.0.1.\n\nThe starter intentionally has one announced quantity bug. Expected baseline: six checks pass and the quantity-total check fails. Select two dosas in the app: the intended subtotal is ₹80.\n\n## Guided task\n\n1. Reproduce the failing test and browser behavior.\n2. Read AGENTS.md and ask the assistant for a short plan.\n3. Request a minimal repair in cart.js, preserving validation and existing tests.\n4. Read the actual diff, then run `npm test` again.\n5. Repeat the original browser interaction. Two dosas plus one tea should total ₹100.\n6. Add one small feature only after the baseline works. Suggested extension: a Clear cart action, with a checked empty-cart behavior.\n7. Update this README with the feature and the checks you actually ran.\n\nUse TASK_PROMPTS.md from the lab pack for copyable prompts. If you apply a browser assistant’s patch manually, record that route accurately.\n\n## Evidence to save\n\nKeep the reviewed diff, before/after test output, one checked UI interaction, a short explanation of price × quantity and your actual client/provider/model route. Do not include keys, login tokens or personal project files.\n\n## Timing\n\n10 minutes reproduce, 10 plan, 20 repair, 15 review/tests, 15 small feature/README, 5 save evidence. Installation belongs in pre-work.\n', 'server.mjs': "import http from 'node:http';\nimport fs from 'node:fs/promises';\nimport path from 'node:path';\nimport {fileURLToPath} from 'node:url';\nconst root = path.dirname(fileURLToPath(import.meta.url));\nconst files = new Map([\n  ['/', ['index.html', 'text/html; charset=utf-8']],\n  ['/index.html', ['index.html', 'text/html; charset=utf-8']],\n  ['/app.js', ['app.js', 'text/javascript; charset=utf-8']],\n  ['/cart.js', ['cart.js', 'text/javascript; charset=utf-8']],\n  ['/styles.css', ['styles.css', 'text/css; charset=utf-8']],\n]);\nconst port = Number(process.env.PORT || 8085);\nif (!Number.isInteger(port) || port < 1024 || port > 65535) throw new Error('Choose a valid development port.');\nhttp.createServer(async (request, response) => {\n  const entry = files.get(new URL(request.url, 'http://127.0.0.1').pathname);\n  if (request.method !== 'GET' || !entry) { response.writeHead(404); response.end('Not found'); return; }\n  try { response.writeHead(200, {'Content-Type': entry[1], 'Cache-Control': 'no-store'}); response.end(await fs.readFile(path.join(root, entry[0]))); }\n  catch { response.writeHead(500); response.end('Could not read the local app file.'); }\n}).listen(port, '127.0.0.1', () => console.log(`BGS Food Lab: http://127.0.0.1:${port} (local development only)`));\n", 'styles.css': ':root{color-scheme:light;--paper:#fdfcfc;--surface:#f5f3f1;--line:#ebe8e4;--ink:#000;--body:#44403b;--muted:#777169;--blue:#0447ff;--orange:#ff4704}\n*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--body);font:18px/1.5 Arial,sans-serif}header{display:flex;justify-content:space-between;align-items:center;padding:27px max(5vw,26px);border-bottom:1px solid var(--line);font-size:14px}header span{color:var(--muted)}.brand{font-size:21px;font-weight:700;color:var(--ink);text-decoration:none}main{max-width:1180px;margin:auto;padding:45px 30px 60px}.eyebrow{font-size:12px;letter-spacing:.13em;color:var(--muted);margin:0 0 12px}h1,h2{font-family:Georgia,serif;font-weight:400;letter-spacing:-.045em;color:var(--ink)}h1{font-size:58px;line-height:1.05;margin:0 0 20px}h2{font-size:30px;line-height:1.2;margin:0}.intro>p:last-child{max-width:590px;margin:0 0 42px;font-size:18px}.workspace{display:grid;grid-template-columns:1.25fr 1fr;gap:66px}.section-title{display:flex;align-items:baseline;justify-content:space-between;margin-bottom:20px}.section-title span{font-size:13px;color:var(--muted)}.menu-row{display:grid;grid-template-columns:1fr auto;gap:22px;padding:23px 0;border-top:1px solid var(--line)}.item-name{font-size:23px;color:var(--ink);margin:0}.price{margin-top:4px;color:var(--muted);font-size:15px}.menu-row label{display:flex;align-items:center;gap:11px;font-size:13px;color:var(--muted)}input{width:66px;font:19px Arial,sans-serif;padding:10px 7px;color:var(--ink);background:#fff;border:1px solid var(--line);border-radius:5px}aside{background:var(--surface);padding:28px 30px;align-self:start}.cart-line{display:flex;justify-content:space-between;gap:20px;font-size:16px;margin:17px 0}.empty{font-size:16px;color:var(--muted);margin:26px 0}.total{display:flex;align-items:baseline;justify-content:space-between;border-top:1px solid #ddd6cf;margin-top:24px;padding-top:21px}.total strong{font-weight:400;font-family:Georgia,serif;color:var(--ink);font-size:42px}.hint{font-size:13px;color:var(--muted);line-height:1.5;margin:18px 0 0}button{width:100%;padding:14px 18px;border:0;border-radius:4px;background:var(--blue);color:white;font:16px Arial,sans-serif;cursor:pointer;margin-top:23px}button:hover{background:#0038d5}button:focus-visible,input:focus-visible,a:focus-visible{outline:3px solid var(--orange);outline-offset:4px}button:disabled{opacity:.5;cursor:default}#summary{font-size:15px;margin:18px 0 0}#error{color:#b32600;font-size:15px}footer{padding:22px 30px;border-top:1px solid var(--line);font-size:12px;color:var(--muted);text-align:center}@media(max-width:760px){header{gap:20px}.workspace{grid-template-columns:1fr;gap:30px}h1{font-size:44px}main{padding-top:34px}}\n', 'TASK_PROMPTS.md': '# Session 5 task prompts\n\nUse one chosen coding route in the synthetic starter folder. Keep normal file/command approvals enabled. These prompts do not authorise unrelated project changes, credentials, external deployment or real ordering.\n\n## Inspect and plan\n\nRead the project instructions, cart.js and cart.test.js. Run the existing tests and inspect the quantity failure. Do not edit yet. Explain why two ₹40 dosas should produce an ₹80 subtotal, then propose a short minimal repair plan. Add no dependencies and keep existing validation and tests.\n\n## Repair the announced bug\n\nApply the focused quantity repair in cart.js. Show the exact diff. Run npm test and report the result you actually observe. Do not remove tests or change expected values to make the result pass. Ask before changes outside this workshop folder.\n\n## Review independently\n\nReview the current diff against the requirement that each line totals unit price multiplied by quantity. Check that invalid quantities, unknown items and empty carts still behave as tested. Run npm test. Report concrete issues with file references and observed evidence; do not edit as part of this review.\n\n## Add one small feature\n\nAfter the baseline repair passes, add a Clear cart action to the page. It should set all quantities to zero, show the existing empty-cart message, set the total to ₹0 and clear the simulated-order draft. Keep keyboard access and existing checks. Describe how you will test the behavior before editing, then show the diff and run the checks you can actually perform. Add no dependency for this feature.\n\n## Explain and document\n\nUpdate the README with the feature, the exact start/test commands and the results you actually observed. Include one known limitation of this toy app. Do not claim that the project places real orders or that an unexecuted browser test passed.\n\n## Manual browser-assistant route\n\nIf the coding client cannot run, paste only the supplied synthetic cart.js and tests into your browser assistant. Ask for a minimal unified diff that repairs quantity handling without changing tests or dependencies. Read the patch, apply it yourself in this workshop folder, then run npm test locally. Record that you applied the patch manually.\n\n## Thirty-second explanation\n\nShow the original failure, the changed multiplication, the same tests passing and a browser interaction with two dosas plus one tea. Explain why that totals ₹100. State the actual client/provider/model route without showing credentials.\n', 'SETUP_HANDOUT.md': '# Session 5 coding setup\n\nChoose one preflighted route before the workshop. Use a new folder containing only the synthetic food project. A client, model and provider are different parts of the setup. A free client or gateway does not provide unlimited inference.\n\n## Shared project setup\n\n1. Install Node.js 20 or later through the official Node website or your college-approved route. Confirm `node --version` and `npm --version`.\n2. Extract food-ordering-starter and open a terminal in that folder.\n3. Run `npm test`. The expected announced baseline is six passing checks and one quantity check failing.\n4. Run `npm start`. Open http://127.0.0.1:8085 in your browser. The development server binds only to your machine. Use a second terminal for your assistant and tests.\n5. Use TASK_PROMPTS.md to inspect, plan, repair and verify. No `npm install` is needed for this starter because it has no package dependencies.\n\nKeep credentials out of project files, instruction files, prompts and shared screenshots. Keep command approvals enabled. Do not use an employer repository or personal files for this exercise.\n\n## Route A Gemini CLI with eligible access\n\nOfficial instructions: https://geminicli.com/docs/get-started/installation/ and https://geminicli.com/docs/get-started/authentication/\n\nCurrent source check on 31 October 2026: Google\'s pages say unpaid-tier and Google One consumer users moved to Antigravity CLI on 18 June 2026. They still document Gemini CLI installation and Gemini API-key/eligible account authentication. Check the route for your actual account before class rather than assuming an older free-sign-in video still applies. Ask the instructor for the preflighted alternative if your account follows the newer flow.\n\nThe documented Gemini CLI installation is:\n\n```powershell\nnpm install -g @google/gemini-cli\ngemini --version\n```\n\nIn the workshop folder, start `gemini` and follow the current official authentication flow supported by your account. If using your own eligible Gemini API key, load it privately for this terminal:\n\n```powershell\n$workshopKey = Read-Host "Your Gemini API key" -AsSecureString\n$env:GEMINI_API_KEY = [Net.NetworkCredential]::new("", $workshopKey).Password\ngemini\n```\n\nSelect the documented Gemini API-key option. Do not type the key into a code file. The chosen account\'s quota and billing apply. After closing the assistant, close the credential terminal or clear its temporary environment variable. Record the actual model and run the food project\'s edit/test cycle. A greeting alone is insufficient.\n\n## Route B OpenCode with local Ollama\n\nOfficial instructions: https://docs.ollama.com/integrations/opencode and https://opencode.ai/docs/providers/\n\nInstall Ollama and OpenCode from their own official instructions for your operating system. The official Ollama integration starts with:\n\n```text\nollama launch opencode\n```\n\nChoose the instructor-tested, genuinely local, tool-capable model. Ollama\'s OpenCode guide requires 64K or higher context. Check the model licence, download size, memory use and speed on the actual laptop before class. Do not download several large models during the session.\n\nOllama also offers cloud models. Selecting one sends requests to its provider. A local terminal is not proof that inference is local. Colab localhost belongs to the cloud runtime, not your laptop\'s Ollama service. Keep this local coding project on the laptop.\n\nIf the local model is slow or unreliable with tool calls, use the prepared API or manual-patch route while retaining the same project and tests.\n\n## Optional instructor route Claude Code through OmniRoute\n\nOmniRoute is third-party community software. Anthropic does not endorse or audit the gateway and does not support routing Claude Code to non-Claude models. This is an optional compatibility demonstration, not a promise of free Claude inference or feature parity.\n\nReviewed source commit: `55ccad495c4a73eb64deaf094829fe5b90ded80c`.\n\n- https://github.com/cheaper-inference/OmniRoute/blob/55ccad495c4a73eb64deaf094829fe5b90ded80c/docs/guides/CLI-INTEGRATIONS.md\n- https://github.com/cheaper-inference/OmniRoute/blob/55ccad495c4a73eb64deaf094829fe5b90ded80c/docs/guides/CLAUDE-CODE-CONFIGURATION.md\n- https://code.claude.com/docs/en/llm-gateway\n\nUse the instructor-tested package version and official Claude Code installation. Before changing an existing client configuration, back up `~/.claude/settings.json` and `~/.claude.json` privately if present; do not include backups in the project or submission.\n\nPowerShell, local-only server:\n\n```powershell\nnpm install -g omniroute\n$env:OMNIROUTE_SERVER_HOST = "127.0.0.1"\nomniroute serve --port 20128 --no-open\n```\n\nKeep this server terminal open. In the printed local dashboard, connect one permitted provider, choose a tool-capable model and create a router inference key in Endpoints. The router key is different from the upstream provider key. Copy a real model ID from the live catalogue.\n\nIn a second terminal, load the router key privately, then launch the client:\n\n```powershell\n$routerKey = Read-Host "OmniRoute inference key" -AsSecureString\n$env:OMNIROUTE_API_KEY = [Net.NetworkCredential]::new("", $routerKey).Password\nomniroute run claude --model "LIVE_MODEL_ID" --api-key-env OMNIROUTE_API_KEY\n```\n\nReplace LIVE_MODEL_ID with the exact current catalogue identifier. The generic `run claude` launcher is current; the older `omniroute launch` remains a supported legacy command. The Claude gateway root has no manually appended `/v1` suffix; use the launcher\'s generated configuration instead of copying URL recipes between products.\n\nInspect Claude Code status and compare it with the router\'s actual upstream request record. A Sonnet-labelled slot can route to another model. Report that actual provider/model and perform a file edit plus the unchanged cart tests. Do not show keys or use account rotation to evade limits.\n\n## Optional instructor route Claude Code through 9Router\n\n9Router is a separate third-party gateway with its own configuration behavior. Use one router at a time; these examples share port 20128.\n\nReviewed source commit: `1fe586bb1cf35e1ed0cb2150a8ef7d1db040b21d`.\n\n- https://github.com/decolua/9router/tree/1fe586bb1cf35e1ed0cb2150a8ef7d1db040b21d\n- https://9router.github.io/en/\n- https://github.com/decolua/9router/blob/1fe586bb1cf35e1ed0cb2150a8ef7d1db040b21d/cli/cli.js\n\nBack up the same existing Claude configuration files privately before changing them. Start in the foreground with explicit loopback binding:\n\n```powershell\nnpm install -g 9router\n9router --host 127.0.0.1 --port 20128 --no-browser\n```\n\nOpen http://127.0.0.1:20128/dashboard. Connect a permitted provider, verify model/tool support, map the Claude Code slots in CLI Tools and Apply. Reopen Claude Code. Check its status against the router/provider\'s actual request record and privately confirm the intended authentication route.\n\nApply can preserve an existing authentication token. Reset removes selected settings and does not reconstruct all previous values; use the private backups when restoring an earlier setup. The reviewed tray restart can omit the local-only host argument, so keep the foreground demo open until the selected release is rehearsed.\n\nDo not copy OmniRoute\'s manual base-URL rule into 9Router. Use the project\'s generated settings and confirm behavior with the exact reviewed version. The same Anthropic non-Claude support limitation applies.\n\n## One check for every route\n\nRun the starter tests, reproduce the bug, request the minimal repair, inspect the actual diff, run the unchanged tests and repeat the UI trigger. The reviewed solution passes all seven checks; this validates the project exercise, not an unexecuted model or gateway integration.\n\nIf a provider returns an auth, quota or model error, stop and identify that layer. Do not claim unlimited credits or turn a canned response into a successful live run. Use the manual patch route when needed and name it in the evidence.\n\n## Tutorial references\n\n- Rahul\'s OmniRoute tutorial: https://www.youtube.com/watch?v=jNzXGJQ__Oo\n- Rahul\'s 9Router tutorial: https://www.youtube.com/watch?v=vtryZ5J6rls\n\nThe videos provide workflow context. Current project documentation and source determine the commands. No coding client or router was installed or run while these materials were prepared; the instructor must rehearse the chosen version, provider and representative student laptop before class.\n'}
# Put every exercise file inside one dedicated workshop folder.
PROJECT = Path.cwd() / "bgs-food-lab"
# Create the folder if needed; an existing folder is allowed.
PROJECT.mkdir(exist_ok=True)
# Count new files so a rerun can explain what it changed.
created = 0
# Only create missing files; rerunning setup preserves your saved work.
for name, source in ASSETS.items():
    # Build the full path for this one starter file.
    path = PROJECT / name
    # Only write a starter file that is not already on disk.
    if not path.exists():
        # Save the supplied text using UTF-8 so currency symbols remain readable.
        path.write_text(source, encoding="utf-8")
        # Increase the count after a missing file has been created.
        created += 1
# Remember the supplied test fingerprint before accepting any later repair.
ORIGINAL_TEST_HASH = hashlib.sha256(ASSETS["cart.test.js"].encode("utf-8")).hexdigest()
# Show which Node version and project folder this run is using.
print(f"Node {node_version}. Project: {PROJECT}")
# Report new files while making clear that existing edits were kept.
print(f"Created {created} files; existing student edits were preserved.")
# Explain that these local project checks do not call an AI model.
print("No npm installation, API key or model request is needed for the checks below.")

# Reuse the original tests for before-and-after evidence.
def run_tests():
    # Fingerprint the current test file before executing it.
    current_hash = hashlib.sha256((PROJECT / "cart.test.js").read_text(encoding="utf-8").encode("utf-8")).hexdigest()
    # Stop if the reference tests were edited instead of the application being repaired.
    if current_hash != ORIGINAL_TEST_HASH:
        # Reject an altered reference test rather than accepting easier checks.
        raise RuntimeError("The original cart tests changed. Restore them from ASSETS before accepting a repair; add new feature checks separately.")
    # Run the same Node tests in the project folder with a 30-second timeout.
    result = subprocess.run([NODE, "--test-reporter=tap", "--test", "cart.test.js"], cwd=PROJECT,
                            capture_output=True, text=True, encoding="utf-8", timeout=30)
    # Display the normal test output so the failure can be inspected.
    print(result.stdout)
    # Check whether Node also wrote an error message.
    if result.stderr:
        # Display error output instead of silently hiding it.
        print(result.stderr)
    # Extract one summary count from the Node test report.
    def count(label):
        # Find the requested total, pass or fail count in the report.
        match = re.search(rf"^# {label} (\d+)", result.stdout, flags=re.MULTILINE)
        # Return a number when the label exists; otherwise mark it as missing.
        return int(match.group(1)) if match else None
    # Return both the counts and raw output for later comparison and saving.
    return {"returncode": result.returncode, "tests": count("tests"), "passed": count("pass"),
            "failed": count("fail"), "stdout": result.stdout, "stderr": result.stderr}


## 2. Predict, then reproduce the failure

One dosa costs ₹40 and tea costs ₹20. What should **two dosas and one tea** cost?

Run the original checks. On a fresh starter, six pass and the quantity check fails. Read the expected and actual subtotal before asking an assistant to change anything. The tests are supposed to reveal the bug.


**Record the baseline**

**What:** Run the original cart tests and save the first observed output.

**Why:** A failing baseline gives you a repeatable repair target.

**Expected:** On a fresh project, seven tests with the one announced quantity failure; repaired copies can pass.

**Read the code:**
1. Call the protected test helper.
2. Save the first baseline output only once.
3. Compare actual counts with the announced starter failure.


In [ ]:
# Observe the actual test result before attempting a repair.
baseline = run_tests()
# Choose the file that will keep the original test evidence.
baseline_log = PROJECT / "baseline-test-output.txt"
# Preserve the first baseline log when this cell is rerun.
if not baseline_log.exists():
    # Save normal and error output together before editing the application.
    baseline_log.write_text(baseline["stdout"] + baseline["stderr"], encoding="utf-8")
# State the expected fresh-starter result; compare it with the observed output.
print("First-run expectation: 7 tests, 6 pass, 1 announced quantity failure.")
# Explain why rerunning a repaired project can give a different baseline.
print("If you have already repaired this saved project, all 7 may now pass.")
# Show the actual counts and exit code from this run.
print("Observed:", {k: baseline[k] for k in ["tests", "passed", "failed", "returncode"]})


## 3. See the same bug in the app

The preview below bundles the current HTML, CSS and JavaScript into a sandboxed frame. It starts no public server and needs no tunnel or account connection.

Set Dosa quantity to 2 and Tea quantity to 1. The fresh starter incorrectly shows ₹60. Explain why the intended total is ₹100. Click **Show simulated order** and compare its draft with the total.


**Preview the current app**

**What:** Combine the current project files into a notebook iframe.

**Why:** Connect the test failure with the quantity behavior you can see.

**Expected:** An inline food-app preview; changing a file requires rerunning the preview.

**Read the code:**
1. Read HTML, CSS and JavaScript from disk.
2. Inline the supplied modules into one page and display an iframe.
3. Try quantity inputs; rerun this preview cell after file edits.


In [ ]:
# @title Open the current app preview
# Define a helper that combines the existing app files into one preview page.
def make_preview_html():
    # Read the page structure from the current index.html file.
    page = (PROJECT / "index.html").read_text(encoding="utf-8")
    # Read the styles that control the current page appearance.
    css = (PROJECT / "styles.css").read_text(encoding="utf-8")
    # Read the current cart calculation code, including any student edits.
    cart = (PROJECT / "cart.js").read_text(encoding="utf-8")
    # Read the current JavaScript that connects page controls to the cart.
    app = (PROJECT / "app.js").read_text(encoding="utf-8")
    # Adapt only the supplied module syntax for this inline preview.
    cart = re.sub(r"^export\s+", "", cart, flags=re.MULTILINE)
    # Remove the one supplied cart import because both scripts will be inlined together.
    app = re.sub(r"^import\s+.*?from\s+['\"]\./cart\.js['\"];?\s*$", "", app, flags=re.MULTILINE)
    # Stop rather than pretending this preview supports additional module imports.
    if re.search(r"^\s*(?:import|export)\s", cart + "\n" + app, flags=re.MULTILINE):
        # Stop on unsupported imports so the preview does not misrepresent the app.
        raise RuntimeError("The inline preview supports the supplied cart.js/app.js modules. Use the laptop project for additional modules.")
    # Replace the external stylesheet link with the CSS text.
    page = page.replace('<link rel="stylesheet" href="styles.css">', '<style>' + css + '</style>')
    # Put cart logic and page behavior in the preview in their required order.
    page = page.replace('<script type="module" src="app.js"></script>', '<script type="module">' + cart + "\n" + app + '</script>')
    # Keep the preview brand link inside the iframe instead of navigating to a server root.
    page = page.replace('class="brand" href="/"', 'class="brand" href="#"')
    # Return the complete page text for the display helper.
    return page

# Build a fresh preview from the files currently on disk.
def show_preview():
    # Rebuild the preview from disk so it reflects the latest saved edits.
    current = make_preview_html()
    # Display the app in an iframe; escape quotes before adding the page to srcdoc.
    display(HTML('<iframe title="Current BGS food app" sandbox="allow-scripts" style="width:100%;height:820px;border:1px solid #ebe8e4" srcdoc="' + html.escape(current, quote=True) + '"></iframe>'))
    # Remind the reader that a saved file change needs a refreshed preview.
    print("Preview uses the current project files. Run this cell again after an edit.")
    # Return the page text so the notebook can keep the preview that was shown.
    return current

# Build and display the initial app before repairing its announced bug.
preview_before = show_preview()


## 4. Ask for a small plan

Read the three synthetic project files printed below. You may share these supplied files with your selected assistant. Keep personal code, keys and login tokens out of the prompt.

Copy this task prompt:

> Two ₹40 dosas should total ₹80, and two dosas plus one ₹20 tea should total ₹100. Read the project instructions, cart.js and cart.test.js. Reproduce the failure and explain a minimal quantity-handling repair before editing. Add no dependencies. Preserve the existing validation and tests. Show the diff and run the same tests.

A browser assistant can propose a patch that you apply manually. A laptop coding agent can inspect the downloaded folder directly. This notebook does not claim either client has run.


**Read before editing**

**What:** Print the project guidance, cart logic and original tests.

**Why:** The requirement and checks should guide a small, reviewable patch.

**Expected:** Three readable files; identify the subtotal calculation before changing anything.

**Read the code:**
1. Read the project instruction file.
2. Read the cart implementation and the original tests.
3. Use the visible requirement and failure to plan a small patch.


In [ ]:
# Read the instructions and test expectations alongside the implementation.
for filename in ["AGENTS.md", "cart.js", "cart.test.js"]:
    # Label each file so instructions, implementation and tests are easy to separate.
    print("\n--- " + filename + " ---\n")
    # Print the real saved file rather than a model-generated description of it.
    print((PROJECT / filename).read_text(encoding="utf-8"))


## 5. Apply the reviewed repair

Edit `cart_source` in the next cell using the patch you reviewed. Change the relevant calculation, then run the cell to save it. Keep the validation and function outputs.

Use this cell as your editor for cart.js. Changing its source persists in your saved notebook. The following cell shows the diff and runs the original tests; it never changes an expected result to hide a failure.


**Apply your reviewed repair**

**What:** Edit the supplied cart source, then write it to cart.js.

**Why:** The starter intentionally contains the announced quantity bug for you to repair.

**Expected:** A saved file; the next cell determines whether the repair works.

**Read the code:**
1. Edit the JavaScript held in cart_source after reviewing your plan.
2. Run the cell to save that exact text as cart.js.
3. The string contains menu data, validation, cart calculation and summary logic; saving it does not repair the exercise automatically.


In [ ]:
# Apply the reviewed change to this source, then run the cell.
# The embedded JavaScript is the editable application source; review your patch before running.
cart_source = r'''export const MENU = Object.freeze({
  dosa: Object.freeze({name: 'Dosa', priceInr: 40}),
  tea: Object.freeze({name: 'Tea', priceInr: 20}),
  idli: Object.freeze({name: 'Idli plate', priceInr: 30}),
});

export function calculateCart(cart) {
  if (!cart || Array.isArray(cart) || typeof cart !== 'object') {
    throw new TypeError('Cart must be an object of item IDs and quantities.');
  }
  const items = [];
  for (const [itemId, quantity] of Object.entries(cart)) {
    if (!Object.hasOwn(MENU, itemId)) throw new RangeError('Unknown menu item.');
    if (!Number.isInteger(quantity) || quantity < 0 || quantity > 10) {
      throw new RangeError('Quantity must be an integer from 0 to 10.');
    }
    if (quantity === 0) continue;
    const item = MENU[itemId];
    const subtotal = item.priceInr;
    items.push({itemId, name: item.name, quantity, unitPriceInr: item.priceInr, subtotalInr: subtotal});
  }
  return {items, totalInr: items.reduce((total, item) => total + item.subtotalInr, 0), currency: 'INR', simulation: true};
}

export function orderSummary(result) {
  return result.items.length
    ? result.items.map(item => `${item.quantity} × ${item.name}`).join(', ') + ` — INR ${result.totalInr} (simulation)`
    : 'Your cart is empty.';
}
'''
# This saves the source you reviewed; saving alone does not prove it is correct.
(PROJECT / "cart.js").write_text(cart_source, encoding="utf-8")
# Point to the next verification step; saving a patch is not a passing test.
print("Saved cart.js. Run the next cell to inspect the diff and tests.")


**Check the repair**

**What:** Show the diff, rerun the original tests and check the quantity example.

**Why:** A repair claim needs both a focused change and observed behavior.

**Expected:** The diff and test output; the quantity assertions run only after all seven tests pass.

**Read the code:**
1. Display and save the actual diff against the starter.
2. Rerun the original tests and save the output.
3. Only after all seven tests pass, check the concrete quantity example from the real module.


In [ ]:
# Compare your current cart logic with the supplied starter.
diff = "".join(difflib.unified_diff(
    ASSETS["cart.js"].splitlines(keepends=True),
    (PROJECT / "cart.js").read_text(encoding="utf-8").splitlines(keepends=True),
    fromfile="starter/cart.js", tofile="your/cart.js"))
# Show the exact patch, or say that the source is still unchanged.
print(diff or "No cart.js change yet.")
# Keep a copy of the reviewed patch alongside the project evidence.
(PROJECT / "reviewed-cart-diff.txt").write_text(diff, encoding="utf-8")
# Run the same protected tests used for the baseline.
after_repair = run_tests()
# Save the after-repair test output for comparison with the baseline.
(PROJECT / "after-test-output.txt").write_text(after_repair["stdout"] + after_repair["stderr"], encoding="utf-8")
# Open the next task only when Node reports success and all seven tests ran.
QUANTITY_CHECK_PASSED = after_repair["returncode"] == 0 and after_repair["tests"] == 7
# Check the concrete food quantities only after the test gate passes.
if QUANTITY_CHECK_PASSED:
    # Ask the real cart module to calculate the concrete two-dosa-and-one-tea example.
    observed = subprocess.run([NODE, "--input-type=module", "-e",
        "import {calculateCart} from './cart.js'; console.log(JSON.stringify(calculateCart({dosa:2,tea:1})));"],
        cwd=PROJECT, check=True, capture_output=True, text=True, encoding="utf-8")
    # Convert the Node JSON output into a Python dictionary for checking.
    observed_cart = json.loads(observed.stdout)
    # Check the total from the repaired module for this classroom example.
    assert observed_cart["totalInr"] == 100
    # Also check the dosa line subtotal so a correct total cannot hide a wrong line.
    assert observed_cart["items"][0]["subtotalInr"] == 80
    # Report the example only after both assertions have passed.
    print("Observed: two dosas cost ₹80; two dosas and one tea cost ₹100.")
else:
    # Keep the feature task blocked while the original repair is unfinished.
    print("The baseline repair is unfinished. Review the failure before the feature task.")


### Check the original trigger again

Run the next preview, select two dosas and one tea, and verify ₹100. Read the diff with your partner. Explain how the changed expression handles a different quantity before moving on.


**Repeat the original interaction**

**What:** Refresh the app preview after the cart repair.

**Why:** The visible interaction should agree with the test evidence.

**Expected:** A fresh preview; try two dosas and compare the subtotal with the original symptom.

**Read the code:**
1. Build a fresh iframe from the saved files.
2. Repeat the original quantity interaction yourself.
3. Compare visible behavior with the test evidence.


In [ ]:
# Refresh after the edit so the iframe uses the current files.
preview_after_repair = show_preview()


### One useful extension: Clear cart

After the original seven tests pass, add a **Clear cart** button. Keep the change small.

- All quantity inputs return to zero.
- The existing empty-cart message appears and the total is ₹0.
- A previously shown simulated-order draft clears.
- The button works with the keyboard.

Ask your chosen assistant for a short plan and a focused patch. The next cell contains the original app and HTML so you can apply the patch here. Keep the cart tests unchanged. An instructor solution is kept in the separate teacher notebook.


**Apply a small feature patch**

**What:** Review and edit the app and page strings for the optional feature task.

**Why:** Keep the verified baseline repair before expanding the app.

**Expected:** The app and HTML save only when the quantity-check gate has passed.

**Read the code:**
1. Review app_source for page events and page_source for the matching controls.
2. Add your small feature in those existing file strings.
3. Save both files only after the quantity repair passes; verify the feature separately.


In [ ]:
# Apply your reviewed feature patch to these strings.
# Edit the browser behavior here; preserve the supplied validation and simulation boundary.
app_source = r'''import {MENU, calculateCart, orderSummary} from './cart.js';

const menu = document.querySelector('#menu');
const cartLines = document.querySelector('#cart-lines');
const total = document.querySelector('#total');
const summary = document.querySelector('#summary');
const error = document.querySelector('#error');
const summaryButton = document.querySelector('#summary-button');
const currency = value => new Intl.NumberFormat('en-IN', {style: 'currency', currency: 'INR', maximumFractionDigits: 0}).format(value);

for (const [itemId, item] of Object.entries(MENU)) {
  const row = document.createElement('div');
  row.className = 'menu-row';
  row.innerHTML = `<div><p class="item-name">${item.name}</p><p class="price">${currency(item.priceInr)} each</p></div><label>Qty <input aria-label="${item.name} quantity" data-item="${itemId}" type="number" min="0" max="10" step="1" value="0"></label>`;
  menu.append(row);
}

function renderCart() {
  error.textContent = '';
  summary.textContent = '';
  try {
    const cart = Object.fromEntries([...menu.querySelectorAll('input')].map(input => [input.dataset.item, Number(input.value)]));
    const result = calculateCart(cart);
    cartLines.replaceChildren();
    if (!result.items.length) {
      const empty = document.createElement('p');
      empty.className = 'empty';
      empty.textContent = 'Your cart is empty. Choose an item to begin.';
      cartLines.append(empty);
    }
    for (const item of result.items) {
      const line = document.createElement('div');
      line.className = 'cart-line';
      line.innerHTML = `<span>${item.quantity} × ${item.name}</span><span>${currency(item.subtotalInr)}</span>`;
      cartLines.append(line);
    }
    total.textContent = currency(result.totalInr);
    summaryButton.disabled = result.items.length === 0;
    summaryButton.onclick = () => { summary.textContent = orderSummary(result); };
  } catch (problem) {
    error.textContent = problem.message;
    summaryButton.disabled = true;
  }
}
menu.addEventListener('input', renderCart);
renderCart();
'''
# Keep page controls and their JavaScript selectors consistent.
page_source = r'''<!doctype html>
<html lang="en">
<head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>BGS Food Lab</title><link rel="stylesheet" href="styles.css"></head>
<body>
  <header><a class="brand" href="/">BGS Food Lab</a><span>Session 5 · Coding with AI</span></header>
  <main>
    <section class="intro"><p class="eyebrow">YOUR WORKSHOP BUILD</p><h1>A little menu.<br>A useful code change.</h1><p>Choose a few items, inspect the total and keep the evidence behind your repair.</p></section>
    <div class="workspace">
      <section aria-labelledby="menu-title"><div class="section-title"><h2>Today’s menu</h2><span>Prices in INR</span></div><div id="menu"></div><p class="hint">Quantities from 0 to 10. All items and orders are fictional.</p></section>
      <aside aria-labelledby="cart-title"><p class="eyebrow">YOUR CART</p><h2 id="cart-title">Order summary</h2><div id="cart-lines" aria-live="polite"></div><div class="total"><span>Total</span><strong id="total">₹0</strong></div><p id="error" role="alert"></p><button id="summary-button" type="button">Show simulated order</button><p id="summary" aria-live="polite"></p><p class="hint">This exercise creates a draft on this page. It sends no order or payment.</p></aside>
    </div>
  </main>
  <footer>V Naga Sai Rahul · BGS College workshop · Learn the behavior, review the diff, run the tests.</footer>
  <script type="module" src="app.js"></script>
</body></html>
'''
# Preserve the repair gate before writing any feature changes.
if QUANTITY_CHECK_PASSED:
    # Save the reviewed browser behavior only after the repair gate has passed.
    (PROJECT / "app.js").write_text(app_source, encoding="utf-8")
    # Save the matching page controls and labels used by that behavior.
    (PROJECT / "index.html").write_text(page_source, encoding="utf-8")
    # Ask for a visible check because writing HTML and JavaScript does not test them.
    print("Saved current app/HTML source. Verify the browser behavior below.")
else:
    # Keep existing app files intact when the required repair has not passed.
    print("Finish the quantity repair first. Existing app/HTML edits were preserved.")


## 6. Check the feature and its limits

Run the same seven cart checks. Then use the current preview to show a simulated order and clear it. Test the Clear cart button using the keyboard as well as the mouse.

The original Node checks cover cart logic. Add a suitable new check for any extracted feature logic, and keep a manual UI checklist for the button behavior. Report exactly which checks ran.


**Check logic and browser behavior**

**What:** Rerun the cart tests and open the current preview.

**Why:** Cart tests do not automatically verify a new interface button.

**Expected:** Test output plus a preview for the explicit manual checks listed below.

**Read the code:**
1. Rerun cart regression tests.
2. Refresh the app preview.
3. Perform the feature interaction yourself; Node cart tests do not inspect the new button.


In [ ]:
# Keep the original cart regression checks after the feature edit.
final_tests = run_tests()
# Use the current preview for the separate manual interface checks.
preview_after_feature = show_preview()
# Separate automated cart checks from the browser feature that still needs a person.
print("The seven Node tests cover cart logic. They do not automatically test the new button in a browser.")
# Give the exact interaction to reproduce in the preview.
print("In this preview: set two dosas and one tea, show the simulated order, then use Clear cart.")
# Use the next evidence cell only after performing those visible checks.
print("Check the four visible outcomes in the next cell yourself.")


## 7. Save evidence you can explain

Set a UI field to `True` only after performing that check in the preview. Update the route description if you used an external assistant. The notebook records your declaration; it does not pretend to have verified a manual browser action.

Share a 30-second explanation: original failure, reviewed diff, the same tests after repair, one observed UI result, and one limitation.


**Record only observed checks**

**What:** Update the check fields only after performing the listed browser actions.

**Why:** The evidence file should distinguish tests, manual checks and the assistant route actually used.

**Expected:** A JSON evidence record; unchecked fields stay false.

**Read the code:**
1. Change each UI field only after performing that visible check.
2. Name the coding-assistant route you actually used and one application limit.
3. Save tests and reported manual checks as separate evidence fields.


In [ ]:
# Change these only after you actually perform the checks in the preview.
# Leave each value false until you have personally performed that check.
UI_CHECKS = {
    "quantities_reset_to_zero": False,
    "empty_message_and_zero_total": False,
    "simulated_order_draft_cleared": False,
    "button_works_with_keyboard": False,
}
# Report the route that actually ran; the notebook itself does not run a coding client.
ASSISTANT_ROUTE = "Manual edit in Colab; no model client ran in this notebook"
# Record one honest scope limit of this teaching application.
KNOWN_LIMITATION = "Synthetic browser exercise; no real orders, payments or backend."

# Keep automated results and student-reported interface checks separate.
evidence = {
    "node_version": node_version,
    "cart_tests": {k: final_tests[k] for k in ["tests", "passed", "failed", "returncode"]},
    "manual_ui_checks_reported_by_student": UI_CHECKS,
    "assistant_route_reported_by_student": ASSISTANT_ROUTE,
    "known_limitation": KNOWN_LIMITATION,
}
# Save the observed test results and student-reported checks as a readable JSON file.
(PROJECT / "lab-evidence.json").write_text(json.dumps(evidence, indent=2, ensure_ascii=False), encoding="utf-8")
# Display the same evidence that was saved, ready for review.
print(json.dumps(evidence, indent=2, ensure_ascii=False))
# Remind the reader that an unchecked field does not mean a check was performed.
print("Unchecked UI fields remain false. A Node test result does not prove a browser check happened.")


## 8. Keep building on your laptop

For a Colab download, set `DOWNLOAD_NOW = True` in the next cell and run it. Local Jupyter also shows a file link. Extract the archive into a new workshop folder and use the included setup handout if you want a laptop coding-agent route. In that folder, `npm test` runs the same tests and `npm start` serves the local app.

Claude Code, Gemini CLI, OpenCode/Ollama and optional routers are separate laptop setup exercises. Their providers, quota and hardware still need preflight. Colab's temporary runtime is not your laptop's Ollama service. No client, router, provider request or model integration was exercised by this notebook.


**Save your work**

**What:** Create a ZIP containing the allowed project files and recorded evidence.

**Why:** Your notebook runtime is not a permanent project backup.

**Expected:** An archive link; the optional Colab download prompt stays off until you choose it.

**Read the code:**
1. Collect only the permitted project and evidence filenames.
2. Create a ZIP and show its download link.
3. Change DOWNLOAD_NOW only if you want Colab to open a browser download prompt.


In [ ]:
# Choose where the downloadable project ZIP will be created.
archive = Path.cwd() / "BGS_Session_05_Your_Project.zip"
# Archive only the named project and evidence files.
allowed_names = set(ASSETS) | {"baseline-test-output.txt", "after-test-output.txt", "reviewed-cart-diff.txt", "lab-evidence.json"}
# Build a portable copy of the saved project.
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    # Visit only the allowed filenames in a consistent order.
    for name in sorted(allowed_names):
        # Find this allowed file inside the exercise folder.
        file = PROJECT / name
        # Skip an optional evidence file if it has not been created yet.
        if file.is_file():
            # Keep one bgs-food-lab folder inside the ZIP for easy extraction.
            zipped.write(file, "bgs-food-lab/" + name)
# Show the ZIP name so the student knows what to save.
print("Project archive:", archive.name)
# Offer a link to the created archive in the notebook output.
display(FileLink(str(archive)))

# In Colab, set this to True if you want the browser download prompt now.
# Keep the browser download prompt an explicit choice.
DOWNLOAD_NOW = False
# Start a browser download only when the student changes the switch to True.
if DOWNLOAD_NOW:
    # Load the Colab download helper only for this optional action.
    from google.colab import files
    # Ask the browser to download the already-created ZIP file.
    files.download(str(archive))


## Before you leave

Save your notebook copy and project archive. Keep Instructor notebooks and answer keys separate from the student starter. Do not mark a failing or unperformed check as passed.

The inline preview uses the supplied HTML/CSS and two JavaScript modules. For a larger app with additional modules or services, use the downloaded laptop project. The Colab preview is a teaching companion, and the app never submits a real order or payment.
